<a href="https://colab.research.google.com/github/christianpm-hub/Pr-ctica-An-lisis-de-Sentimientos-Optativa/blob/main/Christian_P%C3%A9rez_pr%C3%A1ctica_2_1_sentimientos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Qué es el análisis de sentimiento**

Es una tarea de clasificación de texto. Dado un texto, el modelo decide si la intención utilizada es positiva, negativa o neutral.

No es "**entender**" el texto como lo haría una persona. Es detectar patrones de palabras y contexto que, estadísticamente, se asocian a cada categoría.

Para esta práctica en español, el modelo más adecuado es **RoBERTuito**. Es un modelo entrenado específicamente con **500 millones de tweets en español**, lo que le permite entender jerga, emojis, abreviaturas y texto informal.

Está diseñado para redes sociales, que es donde el análisis de sentimiento tiene más aplicación real. Y funciona bien en tareas de sentimiento, odio e ironía en español.

RoBERTuito no se entrenó desde cero. Es un modelo preentrenado: alguien invirtió muchísimo tiempo y recursos en entrenarlo con textos masivos para que "entienda" el español.

Luego ese modelo base se ajustó (mediante fine-tuning) con un conjunto más pequeño de tweets etiquetados como positivos, negativos o neutros. El resultado es un modelo especializado en análisis de sentimiento en español, listo para usar.

Esto es importante porque explica por qué tú no entrenas nada. Usas un modelo que ya existe, ya está entrenado, y solo lo llamas desde Colab. Es exactamente lo que haría un desarrollador profesional: **no reinventar la rueda, usar lo que ya funciona**.

# **Hugging Face**

Hugging Face es la plataforma donde viven estos modelos. Su función **pipeline** es una abstracción: tú le dices "quiero realizar un análisis de sentimiento" y "usa este modelo", y ella se encarga de todo lo demás (preprocesar el texto, pasarlo por el modelo, devolver el resultado).

Solo se necesita entender que pipeline es el "botón" que conecta el texto con un modelo ya entrenado.

El modelo devuelve dos cosas:

    Una etiqueta: POS, NEG o NEU

    Una confianza (score), un número entre 0 y 1

La confianza no es lo mismo que **"acierto"**. Es la seguridad del modelo en su propia predicción. Un score de 0.99 no garantiza que tenga razón; un score de 0.51 en una frase ambigua es normal y no es un fallo.

El modelo no lee intenciones, lee patrones. El guiño o el signo de interrogación alteran el contexto que el modelo aprendió durante el preentrenamiento, y eso mueve la predicción.

In [2]:
from transformers import pipeline

clasificador = pipeline("sentiment-analysis", model="pysentimiento/robertuito-sentiment-analysis")
# carga un modelo de inteligencia artificial
# entrenado para analizar el sentimiento
# de textos en español, con especial enfoque
# en lenguaje informal y redes sociales

# === este es el texto  que se clasifica ===
texto = "Me encanta este módulo de IA"
# =====================================

resultado = clasificador(texto)
print(resultado)

config.json:   0%|          | 0.00/925 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  435MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/384 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.31M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/167 [00:00<?, ?B/s]

[{'label': 'POS', 'score': 0.9514580368995667}]


In [3]:
# clasifica un texto que sea negativo

texto = "Esto es una mierda"
resultado = clasificador(texto)
print(resultado)

[{'label': 'NEG', 'score': 0.966891884803772}]


In [4]:
# clasifica un texto neutro
texto = "Hola"
resultado = clasificador(texto)
print(resultado)

[{'label': 'NEU', 'score': 0.6267504096031189}]


In [5]:
# clasifica un texto irónico
texto = "A lo mejor es que el niño ha estudiado sabes"
resultado = clasificador(texto)
print(resultado)

[{'label': 'NEU', 'score': 0.5770013928413391}]


In [8]:
# clasifica un texto usando emojis
texto = "Hola 😂"
resultado = clasificador(texto)
print(resultado)

[{'label': 'NEU', 'score': 0.5230321884155273}]


**¿Cómo funciona el análisis de sentimientos? ¿Qué límites tiene?**

Según los resultados que han salido al ejecutar los ejercicios anteriores, podemos ver que cuando entra en juego la condición humana, como los refranes, la ironía, o mismamente los emojis que son expresiones reflejadas nuestras, se ve limitado puesto que no puede analizar la positividad o negatividad de todo eso.

# **Práctica**

Implementa en python un analizador de comentarios. Los comentarios positivos y nuetros serán aceptados, salvo que contengan alguna palabra de la lista negra de palabras. Los comentarios negativos que contengan alguna palabra de la lista negra o que tengan un score de más de 0.5 serán rechazadas.

In [11]:
# Pon aquí el código de tu analizador
!pip -q install pysentimiento

import re
import pandas as pd
from pysentimiento import create_analyzer

# Lista negra de palabras prohibidas
PALABRAS_PROHIBIDAS = [
    "mierda", "pendejo", "idiota", "imbécil",
    "estúpido", "gilipollas", "cabron", "puta"
]

# Cargar el modelo de análisis de sentimiento en español
modelo = create_analyzer(task="sentiment", lang="es")

comentarios = [
    "Gracias por tu ayuda.",
    "Este producto es excelente.",
    "Eres un idiota.",
    "Esto es una mierda.",
    "No me gusta nada este servicio.",
    "La experiencia ha sido fantástica.",
    "El producto es normal.",
    "El servicio es bastante malo.",
    "Buenos dias.",
    "Es horrible"
]

# 4. Analizar los comentarios
resultados = []

for comentario in comentarios:

    # Buscar palabras prohibidas
    palabras = [
        palabra for palabra in PALABRAS_PROHIBIDAS
        if re.search(
            rf"(?<!\w){re.escape(palabra)}(?!\w)",
            comentario,
            re.IGNORECASE
        )
    ]

    # Analizar el sentimiento del comentario
    resultado = modelo.predict(comentario)

    sentimiento = resultado.output
    negatividad = resultado.probas["NEG"]

    palabra_negativa = ""

    # Aplicar las reglas de aceptación y rechazo
    if palabras:
        estado = "Rechazado"
        motivo = "Contiene palabras prohibidas: " + ", ".join(palabras)
        palabra_negativa = ", ".join(palabras)

    elif sentimiento == "NEG" and negatividad > 0.5:
        estado = "Rechazado"
        motivo = "Sentimiento negativo con score superior a 0.5"

        # Buscar términos negativos en el comentario.
        # Si no hay palabras prohibidas, mostrar que el rechazo
        # se debe al análisis de sentimiento.
        palabra_negativa = "No detectada por lista negra (sentimiento negativo)"

    else:
        estado = "Aceptado"
        motivo = (
            "Comentario positivo o neutro, o negatividad "
            "no superior a 0.5"
        )

    # Guardar los resultados
    resultados.append({
        "Comentario": comentario,
        "Sentimiento": sentimiento,
        "Score de negatividad": round(negatividad, 3),
        "Estado": estado,
        "Palabra negativa / motivo": palabra_negativa,
        "Motivo": motivo
    })

tabla = pd.DataFrame(resultados)
display(tabla)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

,Comentario,Sentimiento,Score de negatividad,Estado,Palabra negativa / motivo,Motivo
0,Gracias por tu ayuda.,POS,0.014,Aceptado,,"Comentario positivo o neutro, o negatividad no..."
1,Este producto es excelente.,POS,0.005,Aceptado,,"Comentario positivo o neutro, o negatividad no..."
2,Eres un idiota.,NEG,0.892,Rechazado,idiota,Contiene palabras prohibidas: idiota
3,Esto es una mierda.,NEG,0.963,Rechazado,mierda,Contiene palabras prohibidas: mierda
4,No me gusta nada este servicio.,NEG,0.971,Rechazado,No detectada por lista negra (sentimiento nega...,Sentimiento negativo con score superior a 0.5
5,La experiencia ha sido fantástica.,POS,0.004,Aceptado,,"Comentario positivo o neutro, o negatividad no..."
6,El producto es normal.,NEU,0.088,Aceptado,,"Comentario positivo o neutro, o negatividad no..."
7,El servicio es bastante malo.,NEG,0.969,Rechazado,No detectada por lista negra (sentimiento nega...,Sentimiento negativo con score superior a 0.5
8,Buenos dias.,POS,0.073,Aceptado,,"Comentario positivo o neutro, o negatividad no..."
9,Es horrible,NEG,0.672,Rechazado,No detectada por lista negra (sentimiento nega...,Sentimiento negativo con score superior a 0.5
